# Импорты

In [2]:
from enum import Enum
from tokenizers import Tokenizer
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tokenizers.models import BPE, Unigram, WordLevel, WordPiece
from torch.utils.data import DataLoader

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Инициализация токенизаторов

In [3]:
model_name_1 = "LiquidAI/LFM2.5-230M"
tokenizer_1 = Tokenizer.from_pretrained(model_name_1)
print(f'Длина словаря {model_name_1}: {tokenizer_1.get_vocab_size()}')
print(f'Модель токенизатора {model_name_1}: {tokenizer_1.model}')

Длина словаря LiquidAI/LFM2.5-230M: 64402
Модель токенизатора LiquidAI/LFM2.5-230M: BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<|pad|>":0, "<|startoftext|>":1, "<|endoftext|>":2, "<|fim_pre|>":3, "<|fim_mid|>":4, ...}, merges=[("Ċ", "Ċ"), ("Ċ", "ĊĊ"), ("ĊĊ", "Ċ"), ("Ċ", "ĊĊĊ"), ("ĊĊ", "ĊĊ"), ...])


In [4]:
model_name_2 = "xlnet/xlnet-base-cased"
tokenizer_2 = Tokenizer.from_pretrained(model_name_2)
print(f'Длина словаря {model_name_2}: {tokenizer_2.get_vocab_size()}')
print(f'Модель токенизатора {model_name_2}: {tokenizer_2.model}')

Длина словаря xlnet/xlnet-base-cased: 32000
Модель токенизатора xlnet/xlnet-base-cased: Unigram(unk_id=0, vocab=[("<unk>", 0), ("<s>", 0), ("</s>", 0), ("<cls>", 0), ("<sep>", 0), ...], byte_fallback=False)


# Инициализация моделей

In [5]:
tokenizer_model_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype="auto",
                                               device_map="auto")

Loading weights: 100%|██████████| 132/132 [00:00<00:00, 765.35it/s]


In [6]:
tokenizer_model_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype="auto",
                                               device_map="auto")

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 3085.41it/s]


# Предобработка данных

In [7]:
class TokenizerModel(Enum):
    BPE = "BPE"
    WordPiece = "WordPiece"
    Unigram = "Unigram"

In [8]:
exact_match_dict = {}
model_1_exact_tokens = []
model_2_exact_tokens = []

non_exact_tokens = []

if isinstance(tokenizer_1.model, BPE):
    if isinstance(tokenizer_2.model, Unigram):
        vocab_1 = tokenizer_1.get_vocab()
        vocab_2 = tokenizer_2.get_vocab()
        for key in vocab_1.keys():
            key_candidate = key
            if key[0] == 'Ġ':
                key = key.replace('Ġ', '▁')
            if key in vocab_2.keys():
                exact_match_dict[vocab_1[key_candidate]] = vocab_2[key]
                model_1_exact_tokens.append(vocab_1[key_candidate])
                model_2_exact_tokens.append(vocab_2[key])

exact_match_list = [(token_1, token_2) for token_1, token_2 in exact_match_dict.items()]

In [42]:
non_exact_model_1 = {}
non_exact_model_2 = {}
embedding_layer_1 = model_1.get_input_embeddings()
embedding_layer_2 = model_2.get_input_embeddings()

for token, idx in vocab_1.items():
    if idx not in model_1_exact_tokens:
        non_exact_model_1[idx] = embedding_layer_1.weight[idx]

for token, idx in vocab_2.items():
    if idx not in model_2_exact_tokens:
        non_exact_model_2[idx] = embedding_layer_2.weight[idx]

In [11]:
embedding_layer_1 = model_1.get_input_embeddings()
embedding_layer_2 = model_2.get_input_embeddings()
dataset = []

for token_1, token_2 in exact_match_list:
    embed_1 = embedding_layer_1.weight[token_1]
    # print(embed_1.shape)
    embed_2 = embedding_layer_2.weight[token_2]
    # print(embed_2.shape)
    dataset.append((embed_1, embed_2))

In [12]:
dataset_non_exact_1 = {}
dataset_non_exact_2 = {}

for token in non_exact_tokens:
    embed_1 = embedding_layer_1.weight[token]
    dataset_non_exact_1[token] = embed_1

# Построение пайплайна загрузки данных в модель

In [13]:
class Kind(Enum):
    train = "train"
    test = "test"

TRAIN_RATIO = 0.8

In [14]:
class ExactMatchDataset(Dataset):
    def __init__(self, dataset: list[tuple], train_ratio: float, kind: Kind):
        self.dataset = self._train_test_split(dataset=dataset, train_ratio=train_ratio, kind=kind)
    
    def _train_test_split(self, dataset: list[tuple], train_ratio: float, kind: str):
        upper_limit = int(len(dataset)*train_ratio)
        if kind == Kind.train:
            return dataset[:upper_limit]
        elif kind == Kind.test:
            return dataset[upper_limit:]
        else:
            raise ValueError("Выбран неправильный тип датасета: возможен только train или test")

    def __getitem__(self, idx):
        model_1_embed = self.dataset[idx][0] # input embed
        model_2_embed = self.dataset[idx][1] # output embed
        return model_1_embed, model_2_embed
    
    def __len__(self):
        return len(self.dataset)

In [15]:
train = ExactMatchDataset(dataset, TRAIN_RATIO, Kind.train)
valid = ExactMatchDataset(dataset, TRAIN_RATIO, Kind.test)

train_dataloader = DataLoader(train, batch_size=64, shuffle=True)
valid_dataloader = DataLoader(valid, batch_size=64, shuffle=True)

In [16]:
import torch
import torch.nn as nn

class EmbedProjector(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=input_dim, out_features=num_hidden)
        self.norm_1 = nn.LayerNorm(num_hidden)
        self.activation_1 = nn.ReLU()
        self.layer_2 = nn.Linear(in_features=num_hidden, out_features=output_dim)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.skip(x)
        h = self.layer_1(x)
        h = self.norm_1(h)
        h = self.activation_1(h)
        h = self.layer_2(h)
        return h + residual

In [17]:
len(train[2][1])

768

In [18]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
model = EmbedProjector(input_dim=len(train[2][0]), num_hidden=512, output_dim=len(train[2][1])).to(device)

Using cuda device


In [19]:
from tqdm import tqdm

class ModelManager:
    def __init__(self, 
                 epoch_num: int, 
                 model: EmbedProjector, 
                 train_dataloader: DataLoader, 
                 val_dataloader: DataLoader,
                 device: str):
        self.epoch_num = epoch_num
        self.model = model
        self.model_dtype = self._get_model_dtype(model=model)
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.device = device
        self.optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=1e-4)

    def _get_model_dtype(self, model):
        return next(model.parameters()).dtype
    
    def criterion(self, y_predict, y):
        
        cosine_loss = nn.CosineEmbeddingLoss()
        target = torch.ones(
            y_predict.shape[0],
            device=y_predict.device,
            dtype=y_predict.dtype,
        )
        return cosine_loss(y_predict, y, target)

    def train_and_val_loop(self):
        average_train_loss_arr = {}
        average_val_loss_arr = {}
        average_val_loss_list = {}
        cosine_metric = nn.CosineSimilarity(dim=-1)
        cosine_arr = []
        for idx, epoch in enumerate(range(self.epoch_num)):
            self.model.train()
            total_loss = 0
            cosine_similarity = 0
            train_tqdm = tqdm(self.train_dataloader)
            for x_train, y_train in train_tqdm:
                x_train = x_train.to(device=self.device,
                                     dtype=self.model_dtype)
                y_train = y_train.to(device=self.device,
                                     dtype=self.model_dtype)
                self.optimizer.zero_grad()
                predict = self.model(x_train)
                loss = self.criterion(y_predict=predict, y=y_train)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()*x_train.size(0)

            mean_loss = total_loss/len(self.train_dataloader.dataset)
            average_train_loss_arr[epoch] = mean_loss
            print(f'Эпоха = {epoch+1}\nСредний train_loss = {mean_loss}')

            self.model.eval()
            with torch.inference_mode():
                val_tqdm = tqdm(self.val_dataloader)
                total_loss = 0
                for x_val, y_val in val_tqdm:
                    x_val = x_val.to(device=self.device,
                                    dtype=self.model_dtype)
                    y_val = y_val.to(device=self.device,
                                    dtype=self.model_dtype)
                    predict = self.model(x_val)
                    loss = self.criterion(y_predict=predict, y=y_val)
                    batch_cosine_similarity = cosine_metric(predict, y_val)
                    cosine_similarity += batch_cosine_similarity.sum().item()
                    total_loss += loss.item()*x_val.size(0)
            
            mean_cos = cosine_similarity/len(self.val_dataloader.dataset)
            cosine_arr.append(mean_cos)
            mean_loss = total_loss/len(self.val_dataloader.dataset)
            average_val_loss_arr[epoch] = mean_loss

            print(f'Средний val_loss = {mean_loss}')
            print(f'Среднее косинусное сходство = {mean_cos}')
            if idx != 0:
                if average_val_loss_arr[idx] < average_val_loss_arr[idx-1]:
                    print(f'Качество модели улучшилось на {average_val_loss_arr[idx-1]-average_val_loss_arr[idx]} по лоссу, сохраняем модель')
                    torch.save(self.model.state_dict(), 'embedder_model.pth')
                else:
                    print('Модель не оверперформит, заканчиваю обучение...')
                    break
            print('###########################')

            
            
        
        return average_train_loss_arr, average_val_loss_arr

In [ ]:
model_manager = ModelManager(epoch_num=25,
                             model=model, 
                             train_dataloader=train_dataloader, 
                             val_dataloader=valid_dataloader,
                             device='cuda')

average_train_loss_arr, average_val_loss_arr = model_manager.train_and_val_loop()

In [20]:
model_new = EmbedProjector(input_dim=len(train[2][0]), num_hidden=512, output_dim=len(train[2][1])).to(device)
model_new.load_state_dict(
    torch.load("embedder_model.pth", map_location="cuda")
)
model_new.eval()

EmbedProjector(
  (layer_1): Linear(in_features=1024, out_features=512, bias=True)
  (norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
  (activation_1): ReLU()
  (layer_2): Linear(in_features=512, out_features=768, bias=True)
  (skip): Linear(in_features=1024, out_features=768, bias=True)
)

In [21]:
vocab_1_reverse = {value: key for key, value in vocab_1.items()}
vocab_2_reverse = {value: key for key, value in vocab_2.items()}

In [36]:
test = list(non_exact_model_1.values())
# print(f'{test[:5]=}')
tensor1 = test[0]
tensor2 = test[1]

print(f'{tensor1=}')
print(f'{tensor1.shape=}')
print(f'{tensor2=}')
print(f'{tensor2.shape=}')

print(f'{torch.stack((tensor1, tensor2), 0)}')
print(f'{torch.stack((tensor1, tensor2), 0).shape}')

tensor1=tensor([-0.0112,  0.0369, -0.0121,  ..., -0.0623, -0.0063, -0.0140],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)
tensor1.shape=torch.Size([1024])
tensor2=tensor([-0.0293,  0.0217,  0.0086,  ..., -0.0376,  0.0117,  0.0165],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)
tensor2.shape=torch.Size([1024])
tensor([[-0.0112,  0.0369, -0.0121,  ..., -0.0623, -0.0063, -0.0140],
        [-0.0293,  0.0217,  0.0086,  ..., -0.0376,  0.0117,  0.0165]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<StackBackward0>)
torch.Size([2, 1024])


In [ ]:
def running_topk_mean(query, keys, k=10, batch_size=2048): """ Для каждой строки query считает среднее по top-k косинусных сходств с keys, не храня полную матрицу сходств. query: [N_q, D] (нормализованные) keys: [N_k, D] (нормализованные) returns: [N_q] — средняя близость к top-k соседям в keys """ N_q = query.shape[0] result = torch.empty(N_q, device=query.device, dtype=query.dtype) for start in range(0, N_q, batch_size): end = min(start + batch_size, N_q) q_batch = query[start:end] # [B, D] sims = q_batch @ keys.T # [B, N_k] — временная, но маленькая по B topk_vals = sims.topk(k, dim=1).values # [B, k] result[start:end] = topk_vals.mean(dim=1) del sims, topk_vals torch.cuda.empty_cache() return result def csls_best_match(src_embeds, tgt_embeds, k=10, batch_size=2048): src = F.normalize(src_embeds, dim=-1) tgt = F.normalize(tgt_embeds, dim=-1) # r_S(y): близость каждого tgt-токена к его top-k соседям среди src r_tgt = running_topk_mean(tgt, src, k=k, batch_size=batch_size) # [N_tgt] # r_T(x): близость каждого src-токена к его top-k соседям среди tgt r_src = running_topk_mean(src, tgt, k=k, batch_size=batch_size) # [N_src] best_idx = torch.empty(src.shape[0], dtype=torch.long, device=src.device) best_score = torch.empty(src.shape[0], device=src.device, dtype=src.dtype) for start in range(0, src.shape[0], batch_size): end = min(start + batch_size, src.shape[0]) s_batch = src[start:end] # [B, D] sims = s_batch @ tgt.T # [B, N_tgt] csls_batch = 2 * sims - r_src[start:end].unsqueeze(1) - r_tgt.unsqueeze(0) vals, idx = csls_batch.max(dim=1) best_idx[start:end] = idx best_score[start:end] = vals del sims, csls_batch torch.cuda.empty_cache() return best_idx, best_score

In [106]:
# Получаем предсказания от модели по эмбеддингам несовпавших токенов

non_exact_model_1_after_model = {}

for token, embed in non_exact_model_1.items():
    projected_embed = model_new(embed.to(device='cuda', dtype=next(model_new.parameters()).dtype))
    non_exact_model_1_after_model[token] = projected_embed

source_emb = torch.stack(list(non_exact_model_1_after_model.values()))
target_emb = torch.stack(list(non_exact_model_2.values()))

In [107]:
import torch
import torch.nn.functional as F

def running_topk_mean(query, keys, k=10, batch_size=2048):
    """
    Для каждой строки query считает среднее по top-k косинусных сходств
    с keys, не храня полную матрицу сходств.
    query: [N_q, D] (нормализованные)
    keys:  [N_k, D] (нормализованные)
    returns: [N_q] — средняя близость к top-k соседям в keys
    """
    N_q = query.shape[0]
    result = torch.empty(N_q, device=query.device, dtype=query.dtype)

    for start in range(0, N_q, batch_size):
        end = min(start + batch_size, N_q)
        q_batch = query[start:end]                # [B, D]
        sims = q_batch @ keys.T                    # [B, N_k] — временная, но маленькая по B
        topk_vals = sims.topk(k, dim=1).values      # [B, k]
        result[start:end] = topk_vals.mean(dim=1)
        del sims, topk_vals
        torch.cuda.empty_cache()

    return result


def csls_best_match(src_embeds, tgt_embeds, k=10, batch_size=2048):
    src = F.normalize(src_embeds, dim=-1)
    tgt = F.normalize(tgt_embeds, dim=-1)

    # r_S(y): близость каждого tgt-токена к его top-k соседям среди src
    r_tgt = running_topk_mean(tgt, src, k=k, batch_size=batch_size)  # [N_tgt]

    # r_T(x): близость каждого src-токена к его top-k соседям среди tgt
    r_src = running_topk_mean(src, tgt, k=k, batch_size=batch_size)  # [N_src]

    best_idx = torch.empty(src.shape[0], dtype=torch.long, device=src.device)
    best_score = torch.empty(src.shape[0], device=src.device, dtype=src.dtype)

    for start in range(0, src.shape[0], batch_size):
        end = min(start + batch_size, src.shape[0])
        s_batch = src[start:end]                    # [B, D]
        sims = s_batch @ tgt.T                       # [B, N_tgt]
        csls_batch = 2 * sims - r_src[start:end].unsqueeze(1) - r_tgt.unsqueeze(0)
        vals, idx = csls_batch.max(dim=1)
        best_idx[start:end] = idx
        best_score[start:end] = vals
        del sims, csls_batch
        torch.cuda.empty_cache()

    return best_idx, best_score

In [110]:
tokens_1 = list(non_exact_model_1_after_model.keys())
tokens_2 = list(non_exact_model_2.keys())

source_emb = torch.stack([
    non_exact_model_1_after_model[token_id].squeeze(0)
    for token_id in tokens_1
]).detach()

target_emb = torch.stack([
    non_exact_model_2[token_id].squeeze(0)
    for token_id in tokens_2
]).detach()

target_emb = target_emb.to(
    device=source_emb.device,
    dtype=source_emb.dtype,
)

In [111]:
best_idx, best_score = csls_best_match(
    source_emb,
    target_emb,
    k=10,
    batch_size=1024,
)

for i, t1 in enumerate(tokens_1):
    target_row = best_idx[i].item()
    t2 = tokens_2[target_row]

    print(
        f"{vocab_1_reverse[t1]} -> "
        f"{vocab_2_reverse[t2]}, "
        f"CSLS={best_score[i].item():.4f}")

Professional -> professional, CSLS=0.1667
å°Ĥ -> ▁specializing, CSLS=-0.2401
'id -> Stupid, CSLS=-0.1747
Ð¸ÑĤÐµÐ»ÐµÐ¹ -> ▁7,, CSLS=-0.2202
Ġseit -> ▁casin, CSLS=-0.2427
ĠìŀĪìĿĦ -> ▁1,, CSLS=-0.2670
Ġincarn -> ▁incarnation, CSLS=0.3166
å®« -> palace, CSLS=-0.0724
<|reserved_225|> -> ▁9,, CSLS=-0.0063
è»į -> military, CSLS=0.0220
Ġmediana -> ▁1993,, CSLS=-0.2220
ĠzÃ¡ÅĻÃŃ -> ▁2008,, CSLS=-0.0521
èĪ -> ▁rudder, CSLS=-0.2167
åİĭåĬĽ -> ▁pressured, CSLS=0.0385
``Ċ -> ▁2011,, CSLS=-0.2440
ÑĨÑĸÐ² -> ▁9,, CSLS=-0.0919
Ġdeterm -> ▁Determin, CSLS=0.0241
<|img_row_4_col_4|> -> ▁9,, CSLS=-0.0063
." -> ,”, CSLS=0.0046
embly -> election, CSLS=-0.1604
æ²» -> ▁Governance, CSLS=-0.1887
Ġtect -> ▁tectonic, CSLS=0.3350
Ġnomin -> ▁nominate, CSLS=0.0999
ĠTus -> ▁Kappa, CSLS=-0.0905
TYPE -> MANAGEMENT, CSLS=-0.0464
Ġpubblicato -> ▁reprinted, CSLS=-0.1804
Euro -> euro, CSLS=0.0758
Ġmunicipio -> ▁hamlet, CSLS=-0.1209
ĠLaz -> ▁Riz, CSLS=-0.0404
ì§Ģë¥¼ -> highlands, CSLS=-0.2429
ãĤ¯ãĥŃ -> ▁1997,, CSLS=-0.2481
esa